# Llama 3.2 3B Instruct com quantização INT8 (torchao)

Importa as bibliotecas necessárias e verifica se a GPU está disponível.

In [1]:
!uv pip install --upgrade torchao

'uv' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import torch
import torchao
import transformers
from transformers import TorchAoConfig
from torchao.quantization import Int8WeightOnlyConfig
print("✅ Int8WeightOnlyConfig importado com sucesso!")

if torch.cuda.is_available():
    print(f"\n🖥 GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 VRAM: {vram:.1f} GB")
else:
    print("\n⚠ GPU não disponível! Ative em Runtime → Change runtime type → T4 GPU")

ModuleNotFoundError: No module named 'torchao'

Carrega o tokenizer e o modelo Llama-3.2-3B-Instruct quantizado em INT8.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TorchAoConfig
from torchao.quantization import Int8WeightOnlyConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
quant_config = Int8WeightOnlyConfig()
quantization_config = TorchAoConfig(quant_type=quant_config)

print(f"Carregando {MODEL_ID} com INT8...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
)

print(f"✅ Modelo carregado! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Define a função `chat()`, que monta o prompt no formato de chat e gera a resposta do modelo.

In [ ]:
def chat(pergunta, system="Você é um assistente útil. Responda em português.", max_tokens=256):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": pergunta},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    prompt_len = inputs["input_ids"].shape[-1]
    new_tokens = output_ids[0][prompt_len:]
    resposta = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return resposta

Testa a função `chat` com uma única pergunta.

In [ ]:
print(chat("Explique o que é machine learning em 3 frases."))

Faz um loop com várias perguntas e imprime cada pergunta junto com a resposta do modelo.

In [ ]:
perguntas = [
    "Qual a diferença entre Python e JavaScript?",
    "Me dê 3 dicas para aprender programação.",
    "Resuma o que é uma rede neural.",
]

for p in perguntas:
    print(f"\n{'='*60}")
    print(f"👤 {p}")
    print(f"{'='*60}")
    print(f"🤖 {chat(p)}")